# Pandas transaction analysis

## Learning goals

Inspect a frame, select with `loc` and `iloc`, reason about missing and duplicate rows, aggregate with `groupby`, `merge` reference data, sort deterministically, parse datetime values, transform columns, and use categorical dtypes.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("datasets/transactions.csv")
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError:
        print("Banking dataset not present; completed examples use a tiny synthetic frame.")
    else:
        uploaded = files.upload()
        if uploaded:
            DATA_PATH = Path(next(iter(uploaded)))

In [ ]:
if DATA_PATH.exists():
    transactions = pd.read_csv(DATA_PATH)
else:
    transactions = pd.DataFrame(
        {
            "id": ["demo-1", "demo-2", "demo-2"],
            "amount": [12.50, 88.00, 88.00],
            "currency": ["EUR", "USD", "USD"],
            "merchant": ["Aurora Market", "Fjord Books", "Fjord Books"],
            "category": ["groceries", None, None],
            "timestamp": [
                "2025-01-02T09:00:00+00:00",
                "2025-01-03T10:00:00+00:00",
                "2025-01-03T10:00:00+00:00",
            ],
            "is_fraud": [False, True, True],
        }
    )
transactions["amount"] = pd.to_numeric(transactions["amount"], errors="raise")
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], utc=True)
transactions["category"] = transactions["category"].astype("category")
transactions.head(), transactions.dtypes

In [ ]:
high_value = transactions.loc[transactions["amount"] >= 50, ["id", "amount"]]
first_two_rows = transactions.iloc[:2]
missing_categories = transactions["category"].isna().sum()
duplicate_ids = transactions.duplicated(subset=["id"], keep=False)
high_value, first_two_rows, missing_categories, duplicate_ids

In [ ]:
merchant_risk = pd.DataFrame(
    {"merchant": ["Aurora Market", "Fjord Books"], "risk_band": ["low", "medium"]}
)
enriched = transactions.merge(merchant_risk, on="merchant", how="left", validate="many_to_one")
enriched["day"] = enriched["timestamp"].dt.floor("D")
summary = (
    enriched.groupby(["day", "currency"], observed=True, as_index=False)
    .agg(transaction_count=("id", "size"), total_amount=("amount", "sum"))
    .sort_values(["day", "currency"], kind="stable")
)
summary

In [ ]:
# Exercise 1: report missing values and duplicate transaction IDs without mutating the input frame.
raise NotImplementedError("Complete this notebook exercise")

In [ ]:
# Exercise 2: build a deterministic currency/category groupby with named aggregations.
raise NotImplementedError("Complete this notebook exercise")

In [ ]:
# Exercise 3: merge merchant risk data with many-to-one validation.
# Preserve unmatched transactions.
raise NotImplementedError("Complete this notebook exercise")

## Self-check

1. When should you choose `loc` rather than `iloc`?
2. Why make duplicate handling explicit before aggregation?
3. What does `validate="many_to_one"` protect?
4. Why parse datetime values with `utc=True`?
5. When does a categorical column reduce ambiguity?